# CNN / RNN Training — CMPSC 448
Select **Runtime → Change runtime type → T4 GPU**, set the paths below, and run the cells in order. **Run all trains all six experiments sequentially.** After running the shared setup and helper cells, you can also run any experiment cell individually.

RQ1 compares CNN and RNN using `LLM_output` only. RQ2 compares input only (`LLM_Input`), output only (`LLM_output`), and input + output for each model.

Required files: `train.py` and `data.py` in the project folder, and `cnn.py` and `rnn.py` in the model folder. Imports in `train.py` must match the actual model location (`src` or `model`). Dataset files: `train.csv`, `validation.csv`, and `test.csv`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip -q install torch numpy pandas scikit-learn matplotlib

## Shared Paths and Training Settings
Set `PROJECT_DIR` and `DATA_DIR` to your actual Drive folders. All six experiments use the same data splits, seed, and shared training settings. If GPU memory is insufficient, reduce `BATCH_SIZE` to 32 or 16 before running the experiments.

In [3]:
from pathlib import Path
import sys, subprocess, json
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm')
DATA_DIR = Path('/content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm/results')

EPOCHS = 15
PATIENCE = 3
BATCH_SIZE = 64
MAX_LEN = 512
SEED = 42
# Optional existing vocabulary paths, one per input mode.
VOCAB_PATHS = {'input': None, 'output': None, 'both': None}

for name in ('train.py', 'data.py'):
    assert (PROJECT_DIR / name).is_file(), f'Code file not found: {PROJECT_DIR / name}'
for name in ('train.csv', 'validation.csv', 'test.csv'):
    assert (DATA_DIR / name).is_file(), f'CSV file not found: {DATA_DIR / name}'
for mode, vocab_path in VOCAB_PATHS.items():
    if vocab_path is not None:
        assert Path(vocab_path).is_file(), f'Vocabulary file not found: {vocab_path}'
assert torch.cuda.is_available(), 'Select a GPU runtime and rerun starting from this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('CSV:', DATA_DIR)
print('Results:', OUTPUT_DIR)

# Check the imports and command-line options in train.py before training.
probe = subprocess.run([sys.executable, str(PROJECT_DIR / 'train.py'), '--help'],
                       cwd=str(PROJECT_DIR), capture_output=True, text=True)
if probe.returncode != 0:
    print(probe.stdout, probe.stderr)
    raise RuntimeError('Check the imports in train.py. If the models are in src/, use from src.cnn / from src.rnn.')
print('train.py is ready to run')

GPU: Tesla T4
CSV: /content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm/data
Results: /content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm/results
 Traceback (most recent call last):
  File "/content/drive/MyDrive/PSU Courses/Senior/CMPSC 448/CMPSC-448-Midterm/train.py", line 18, in <module>
    from model.cnn import TextCNN, LABELS, ID2LABEL
ModuleNotFoundError: No module named 'model'



RuntimeError: Check the imports in train.py. If the models are in src/, use from src.cnn / from src.rnn.

## Shared Training and Result Helpers
The best model is selected using validation Macro-F1, then evaluated once on the test set. Each experiment cell streams its training logs and displays Accuracy / Macro-F1, per-class metrics, training curves, the confusion matrix, and sample predictions. Results are saved to Drive in a new run folder.

Keep the runtime active during training. Rerunning the helper cell resets the in-memory record of completed experiments; rerun the experiment cells to populate the final comparison again.

In [ ]:
import pandas as pd
from IPython.display import display, Image

EXPERIMENTS = (
    ('cnn', 'output'), ('rnn', 'output'),
    ('cnn', 'input'), ('rnn', 'input'),
    ('cnn', 'both'), ('rnn', 'both'),
)
RUNS = {}

def run_experiment(model, mode, vocab_path=None):
    experiment_dir = OUTPUT_DIR / f'{model}_{mode}'
    before = set(experiment_dir.glob('*/summary.csv'))
    command = [
        sys.executable, '-u', str(PROJECT_DIR / 'train.py'),
        '--model', model, '--mode', mode,
        '--data-dir', str(DATA_DIR), '--output-dir', str(OUTPUT_DIR),
        '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
        '--batch-size', str(BATCH_SIZE), '--max-len', str(MAX_LEN),
        '--seed', str(SEED), '--device', 'cuda', '--num-workers', '0',
    ]
    if vocab_path is not None:
        command += ['--vocab-path', str(vocab_path)]
    print(f'Starting training: {model} / {mode}', flush=True)
    # An argument list handles paths containing spaces without using a shell command string.
    with subprocess.Popen(command, cwd=str(PROJECT_DIR), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Training failed (exit={returncode}). Check the logs above.')
    created = set(experiment_dir.glob('*/summary.csv')) - before
    assert len(created) == 1, f'Check the results folder for the new run: {experiment_dir}'
    return next(iter(created)).parent

def show_results(run_dir):
    display(pd.read_csv(run_dir / 'summary.csv'))
    metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(metrics['classification_report']).T)
    display(Image(filename=str(run_dir / 'training_and_confusion.png')))
    display(pd.read_csv(run_dir / 'predictions.csv').head())
    print('Saved files:', ', '.join(p.name for p in sorted(run_dir.iterdir())))


def train_and_review(model, mode):
    # Remove any previous result for this experiment before attempting a rerun.
    RUNS.pop((model, mode), None)
    run_dir = run_experiment(model, mode, VOCAB_PATHS[mode])
    RUNS[(model, mode)] = run_dir
    print('Done! Results folder:', run_dir)
    show_results(run_dir)
    return run_dir

## Experiment 1 — CNN: Output Only (RQ1 / RQ2)

In [ ]:
cnn_output_run = train_and_review('cnn', 'output')

## Experiment 2 — RNN: Output Only (RQ1 / RQ2)

In [ ]:
rnn_output_run = train_and_review('rnn', 'output')

## Experiment 3 — CNN: Input Only (RQ2)

In [ ]:
cnn_input_run = train_and_review('cnn', 'input')

## Experiment 4 — RNN: Input Only (RQ2)

In [ ]:
rnn_input_run = train_and_review('rnn', 'input')

## Experiment 5 — CNN: Input + Output (RQ2)

In [ ]:
cnn_both_run = train_and_review('cnn', 'both')

## Experiment 6 — RNN: Input + Output (RQ2)

In [ ]:
rnn_both_run = train_and_review('rnn', 'both')

## Compare All Six Experiments
This cell compares completed experiments recorded in this notebook session and saves `comparison_current.csv`. It lists any missing experiments so a partial table cannot be mistaken for a complete six-experiment comparison. It does not start training or automatically mix in older saved runs.

For RQ1, compare the two output-only rows. For RQ2, compare input, output, and both within each model. Use the results in your report to explain whether prompts improve classification. If input-only performance is high, investigate prompt topic/style differences across labels and duplicate prompts across data splits. Do not select model settings based on test performance.

In [ ]:
tables = []
missing = []
for model, mode in EXPERIMENTS:
    run_dir = RUNS.get((model, mode))
    if run_dir is None:
        missing.append(f'{model} / {mode}')
        continue
    table = pd.read_csv(run_dir / 'summary.csv')
    table['run_dir'] = str(run_dir)
    tables.append(table)

if tables:
    comparison = pd.concat(tables, ignore_index=True)
    display(comparison)
    comparison_path = OUTPUT_DIR / 'comparison_current.csv'
    comparison.to_csv(comparison_path, index=False)
    print('Comparison saved:', comparison_path)
else:
    print('No completed experiments in this session. Run the experiment cells first.')

if missing:
    print(f'Incomplete comparison ({len(tables)}/6 experiments). Missing:', ', '.join(missing))
else:
    print('All six experiments are included.')